##**Quantitative Translational Imaging in Medicine Lab — Summer Scholar Program**

<div style="background-color:black; padding:20px; text-align:center; border-radius: 8px;">
    <img src="https://i0.wp.com/www.martinos.org/wp-content/uploads/2019/01/spark_no_fade.gif?fit=404%2C303&ssl=1" alt="Martinos Center Logo" width="400"/>
    <h1 style="color:white; font-family: sans-serif;">Martinos Center for Biomedical Imaging</h1>
</div>

## Day 4: Welcome back! Connecting this to your first three days

Over the last three days you learned that every medical image — MRI, CT, ultrasound — is
really just a grid of numbers, and that you can write simple rules to find things in that
grid (remember thresholding, where we found "bone" in a CT scan just by checking if a pixel's
number was above 300?).

That threshold rule was written **by us, by hand**. It worked because we already knew what
number range to look for.

But what if we don't know the right rule? What if the pattern that separates "healthy tissue"
from "diseased tissue" is too complex or subtle for a human to write down as a simple rule?

This is exactly where **AI (specifically, deep learning)** comes in. Instead of a human
writing the rule, we show the computer thousands of labeled examples ("this one is healthy,
this one has disease") and let it **learn the rule itself**. That's what you'll do for the
next two days: build, train, and evaluate a real AI model that looks at medical images and
learns to classify them — the same basic category of tool used in real hospitals and research
labs today (and an active area of ongoing research, including here at the Martinos Center).

A note on pace: this material moves faster and covers more ground than the first three days. The "Your Turn" sections are where you'll drive,
with your mentor's help. Ask questions liberally; if something feels like a black box, that's
exactly the right thing to ask about.


# Medical Image Classification: From X-Rays to Digital Pathology

**AI Medical Imaging Workshop**

---

## Course Overview

Welcome to this two-day hands-on workshop on medical image classification. The goal is to take you from the very basics of handling medical images all the way to building your own AI models to detect diseases.

This isn't a typical lecture. The core learning principle here is **"Learn by Example, Apply by Project."**

1.  **Learn with an Example (`ChestMNIST`)**: For each new concept, we will first walk through it together, step-by-step, using a dataset of Chest X-rays called `ChestMNIST`. This is our simple, intuitive example.
2.  **Apply to the Project (`PathMNIST`)**: After each example, it will be **Your Turn**! You will immediately apply what you've learned to our main project dataset, `PathMNIST`, which consists of images of tissue samples. In these sections, you will see `# --- YOUR CODE HERE ---` where you'll need to write the code yourself, using the example as your guide.

Don't worry, we'll guide you with hints and questions along the way. The goal is for you to learn by doing.

### Course Structure

**Day 1: Foundations & Data Exploration**

* **Module 1: Introduction to AI in Medicine & Our Tools**
    * The "Why": The importance of AI in medical diagnosis.
    * The "What": Understanding Chest X-Rays vs. Digital Pathology.
    * Setup: Getting your Python environment ready.
* **Module 2: Loading, Exploring, and Visualizing Medical Images**
    * *Example*: Loading and understanding the `ChestMNIST` dataset.
    * *Project*: It's your turn to load and visualize the `PathMNIST` dataset.
* **Module 3: Data Preprocessing and Augmentation**
    * *Example*: Why and how we preprocess images for an AI.
    * *Project*: It's your turn to build a more advanced preprocessing pipeline for `PathMNIST`.

**Day 2: Building, Training, and Understanding AI Models**

* **Module 4: Building Your First AI Brain (A Neural Network)**
    * Introduction to Convolutional Neural Networks (CNNs).
    * *Example*: Building a simple CNN for `ChestMNIST`.
    * *Project*: It's your turn to adapt the CNN for `PathMNIST`.
* **Module 5: Teaching the AI and Checking Its Work**
    * Understanding the training loop, loss functions, and optimizers.
    * *Example*: Training the `ChestMNIST` model and evaluating its performance.
    * *Project*: It's your turn to train the `PathMNIST` model and analyze its mistakes.
* **Module 6: Advanced AI Techniques & Next Steps**
    * Improving performance with a powerful technique called Transfer Learning.
    * Looking inside the "black box" to see *what* the AI is looking at.
    * Conclusion and where you can go from here.

## Module 1: Introduction & Setup

### The Role of AI in Medical Imaging

Medical imaging is a cornerstone of modern healthcare, producing vast amounts of data from X-rays, CT scans, MRIs, and pathology slides. For a radiologist or pathologist, analyzing these images is a highly skilled but often repetitive and time-consuming task.

This is where Artificial Intelligence, specifically deep learning, can be a revolutionary partner for doctors. AI models can be trained to:
* **Detect** anomalies (like tumors or fractures) quickly.
* **Classify** diseases with high accuracy.
* **Segment** organs or tissues for precise analysis.
* **Prioritize** urgent cases for human experts, acting like a smart assistant.

In this course, we'll focus on **classification**: teaching a machine to look at an image and assign it a specific medical label.

### Meet the Datasets: MedMNIST v2

We will be using datasets from the **MedMNIST v2** collection, a fantastic resource of standardized, beginner-friendly medical image datasets.

1.  **Our Example: `ChestMNIST`**
    * **What it is:** A collection of 2D Chest X-ray images.
    * **The Task:** Multi-label classification of common thoracic diseases.
    * **Why it's a good example:** X-rays are grayscale, relatively low-resolution, and visually intuitive for most people.

2.  **Our Project: `PathMNIST`**
    * **What it is:** A collection of 2D Histopathology images. These are colored microscopic images of tissues sampled from patients.
    * **The Task:** Multi-class classification of colon tissue, identifying benign tissue vs. cancerous subtypes.
    * **Why it's a great project:** It introduces new challenges: color images, subtle textural differences, and the need for a more robust model.

### Environment Setup

Let's get our tools ready. We need `torch` and `torchvision` for building our models, `medmnist` to easily access our datasets, and `matplotlib` for visualization. The first step is always to install the necessary libraries.

In [ ]:
# Install the necessary libraries
# The '!' tells the notebook to run this command in the command line / terminal.
!pip install torch torchvision torchaudio
!pip install medmnist
!pip install matplotlib numpy scikit-learn

In [ ]:
# Import the libraries we'll need throughout the project
# 'import' makes the library's functions available to use in our code.
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

import medmnist
from medmnist import INFO, Evaluator

import matplotlib.pyplot as plt
import numpy as np

print(f"MedMNIST v{medmnist.__version__} is installed.")

---

## Module 2: Loading, Exploring, and Visualizing Medical Images

This is the most critical first step in any AI project. We can't build a model without deeply understanding our data first. It's like a chef needing to know their ingredients before they can cook.

### Part A (Example): Loading and Exploring `ChestMNIST`

First, let's learn the ropes with the Chest X-ray dataset.

#### Technical Step: Downloading and Loading the Data

The `medmnist` library makes this incredibly simple. We'll download the `ChestMNIST` dataset, which is pre-split into `train`, `validation`, and `test` sets.
- **Training set:** Used to teach the model.
- **Validation set:** Used to check the model's performance during training and tune it.
- **Test set:** Used only at the very end to get a final, unbiased score of how good our model is.

In [ ]:
# First, we define a 'transform'. This is a set of instructions for how to prepare the image data.
# For now, our only instruction is transforms.ToTensor(). This converts the images into a format PyTorch understands, called a Tensor.
data_transform = transforms.Compose([
    transforms.ToTensor()
])

# Now, we download and load the datasets. The library handles everything for us.
train_dataset = medmnist.ChestMNIST(split='train', transform=data_transform, download=True)
val_dataset = medmnist.ChestMNIST(split='val', transform=data_transform, download=True)
test_dataset = medmnist.ChestMNIST(split='test', transform=data_transform, download=True)

print("ChestMNIST Datasets:")
print(f"Training set has {len(train_dataset)} images")
print(f"Validation set has {len(val_dataset)} images")
print(f"Test set has {len(test_dataset)} images")

#### Technical Step: Understanding the Data's Shape

What did we just load? A dataset in PyTorch is like a big list where each item is a pair: `(image, label)`.

The image isn't just a picture; it's a **Tensor**. You can think of a Tensor like a multi-dimensional spreadsheet for numbers. For an image, this spreadsheet holds the brightness value of each pixel.

Let's inspect the very first image in our training set to see what this means.

In [ ]:
# Get the first training sample. It returns the image and its corresponding label.
first_image, first_label = train_dataset[0]

# Print the 'shape' or dimensions of the image tensor
print(f"Image shape: {first_image.shape}")

# Print the label tensor
print(f"Label (tensor): {first_label}")

**Analysis of the Output:**

* **`Image shape: torch.Size([1, 28, 28])`**: This is the most important part! It's read as (Channels, Height, Width).
    * **Channels = 1**: This means the image is grayscale. It only has one channel for brightness.
    * **Height = 28**: The image is 28 pixels tall.
    * **Width = 28**: The image is 28 pixels wide.
* **`Label (tensor): [1. 0. 0. ...]`**: The label is a tensor with 14 numbers. This tells us it's a **multi-label** problem. Each of the 14 spots corresponds to a different disease. A `1` means the disease is present, and a `0` means it's absent. A single image can have multiple diseases.

#### Medical Context: What Do the Labels Mean?

A tensor of `1`s and `0`s is meaningless without context. We need to map it back to a medical finding. The `medmnist.INFO` object contains all this metadata.

In [ ]:
# Get information about the ChestMNIST dataset
info = INFO['chestmnist']
label_map = info['label']
print(f"Task: {info['task']}")

print("Label Mapping (what each of the 14 spots in the label tensor means):")
print(label_map)

#### Medical Explanations: Chest Diseases

It's great that we have these labels, but what do they actually mean? Here are simple explanations for some of them and why an AI to detect them is useful:

- **Atelectasis**: This is a complete or partial collapse of the entire lung or an area (lobe) of the lung. It's one of the most common breathing complications after surgery.
- **Cardiomegaly**: This simply means an enlarged heart. It's not a disease itself, but a sign of another condition, like high blood pressure or a heart valve problem. An AI can quickly measure the heart's size on an X-ray to flag it for a doctor.
- **Effusion (Pleural Effusion)**: This is a buildup of excess fluid between the layers of the pleura outside the lungs (the thin membranes that line the lungs and the inside of the chest cavity). It can be caused by many conditions, from heart failure to pneumonia.
- **Pneumonia**: An infection that inflames the air sacs in one or both lungs. The air sacs may fill with fluid or pus. It can be serious, and spotting its signs on an X-ray is a key diagnostic step.
- **Pneumothorax**: A collapsed lung. This occurs when air leaks into the space between your lung and chest wall. This air pushes on the outside of your lung and makes it collapse. It can be a medical emergency.

**Why is this useful?** A radiologist has to look at hundreds of these images a day. An AI can act as a second pair of eyes, highlighting potential findings, prioritizing urgent cases (like a pneumothorax), and helping to reduce errors.

#### Technical Step: Visualizing the Images

Let's look at a few examples to get a feel for the data. We'll write code to find all the positive labels for an image and display their names as the title.

In [ ]:
# A DataLoader is a helper that automatically groups our data into batches.
# This is much more efficient for training than feeding images one-by-one.
train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)

# Get one batch of images and labels from the loader
images, labels = next(iter(train_loader))

# Setup a plot to show 16 images
plt.figure(figsize=(12, 12))
plt.suptitle("ChestMNIST - Chest X-Rays (28x28)", fontsize=16)

# Loop through the first 16 images in the batch
for i in range(16):
    plt.subplot(4, 4, i + 1)
    # The image is [1, 28, 28], so we squeeze it to [28, 28] for plotting
    image = np.squeeze(images[i].numpy())

    # For multi-label, find all the spots in the label tensor that are '1'
    label_tensor = labels[i]
    positive_indices = torch.where(label_tensor == 1)[0]

    # Get the names of the labels from our label_map
    if len(positive_indices) > 0:
        label_names = [label_map[str(idx.item())] for idx in positive_indices]
        title_text = "\n".join(label_names)
    else:
        title_text = "No Finding"

    plt.imshow(image, cmap='gray') # Use a gray colormap for X-rays
    plt.title(title_text)
    plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

---

### Part B (Project): Applying Your Skills to `PathMNIST`

You've successfully loaded, inspected, and visualized the `ChestMNIST` dataset. Now, it's your turn to apply those exact same skills to our main project, `PathMNIST`.

#### Medical Context: What is Digital Pathology?

First, what are we looking at? Unlike an X-ray which images a whole organ system from outside the body, **histopathology** is the microscopic examination of biological tissues.

1.  A tissue sample (biopsy) is taken from a patient (in this case, from the colon).
2.  It's thinly sliced, placed on a glass slide, and stained with special dyes (most commonly Hematoxylin and Eosin, or H&E).
3.  This staining makes different cellular structures visible. For example, the cell nucleus turns purple/blue, and the cytoplasm and extracellular matrix turn pink.
4.  The slide is digitized using a high-resolution scanner.

The resulting image allows a pathologist to identify cancerous cells. Our `PathMNIST` dataset contains small patches from these larger digitized slides.

#### Your Turn: Loading the `PathMNIST` Data

Use the same process as before to download and load the `PathMNIST` dataset. Fill in the `--- YOUR CODE HERE ---` sections.

In [ ]:
# We can use the same simple data_transform for now
data_transform = transforms.Compose([
    transforms.ToTensor()
])

# Task: Create the train, validation, and test datasets for PathMNIST.
# Hint: The dataset class is called medmnist.PathMNIST(...)
path_train_dataset_raw = # --- YOUR CODE HERE ---
path_val_dataset_raw = # --- YOUR CODE HERE ---
path_test_dataset_raw = # --- YOUR CODE HERE ---

# Print their sizes to confirm
print("PathMNIST Datasets:")
print(f"Training set has {len(path_train_dataset_raw)} images")
print(f"Validation set has {len(path_val_dataset_raw)} images")
print(f"Test set has {len(path_test_dataset_raw)} images")

#### Your Turn: Inspecting a `PathMNIST` Sample

Now, get the first data sample from the `path_train_dataset_raw` and print its shape and label. What do you notice that's different from `ChestMNIST`?

In [ ]:
# Task: Get the first sample from the PathMNIST training set.
first_path_image, first_path_label = # --- YOUR CODE HERE ---

# Task: Print the shape of the image tensor.
# --- YOUR CODE HERE ---

# Task: Print the label. Does it look different from the ChestMNIST label?
# --- YOUR CODE HERE ---


**Questions to Answer:**

1.  Look at the image shape. How many channels does this image have? What does that tell you about the image?
2.  Look at the label. Is this a multi-label or multi-class problem? (Hint: multi-class means there's only one correct label per image).

#### Your Turn: Understanding the `PathMNIST` Labels

Now, use the `INFO` object to find out what the `PathMNIST` labels represent.

In [ ]:
# Task: Get the info dictionary for 'pathmnist'
path_info = # --- YOUR CODE HERE ---
path_label_map = path_info['label']
print("PathMNIST Label Mapping:")
print(path_label_map)

#### Medical Explanations: Colon Tissue Types

Just like with the X-rays, these labels represent real biological structures that a pathologist studies under a microscope:

- **Adipose (ADI)**: This is body fat tissue. It's often found adjacent to organs and is a normal finding.
- **Background (BACK)**: These are parts of the slide that don't contain any tissue, like the glass itself or mounting medium.
- **Debris (DEB)**: These are small fragments of cells or other materials that can appear during the slide preparation process.
- **Lymphocytes (LYM)**: A type of immune cell. Seeing a lot of them can indicate inflammation or an immune response, which is common near tumors.
- **Mucus (MUC)**: A normal secretion from the colon lining. However, some types of colon cancer produce large amounts of mucus.
- **Smooth Muscle (MUS)**: This is part of the normal wall of the colon.
- **Normal Colon Mucosa (NORM)**: This is the healthy, normal lining of the colon.
- **Stroma (STR)**: The connective tissue that supports an organ. Cancer cells often interact with the stroma as they grow.
- **Tumor (TUM)**: This is colorectal cancer epithelium. This is the most critical class to identify correctly.

**Why is this useful?** A pathologist's job is to find the tumor cells (TUM) and distinguish them from all the other normal or benign tissues. An AI that can automatically classify these patches can help speed up this process, allowing the pathologist to focus on the most suspicious areas of a large slide.

#### Your Turn: Visualizing the `PathMNIST` Images

Now, write the code to visualize a batch of 16 images.

**Hint:** When plotting a PyTorch color image tensor with shape `[3, H, W]`, `matplotlib` expects the shape to be `[H, W, 3]`. You will need to rearrange the dimensions. The `.permute(1, 2, 0)` method is perfect for this.

In [ ]:
# Task: Create a DataLoader for the path_train_dataset_raw
path_train_loader_raw = # --- YOUR CODE HERE ---

# Task: Get one batch of images and labels
path_images, path_labels = # --- YOUR CODE HERE ---

# --- Visualization Code (Complete this part) ---
plt.figure(figsize=(12, 10))
plt.suptitle("PathMNIST - Colon Pathology (28x28)", fontsize=16)

for i in range(16):
    plt.subplot(4, 4, i + 1)

    # Task: Get the i-th image and permute its dimensions for plotting
    image = # --- YOUR CODE HERE --- .permute(1, 2, 0).numpy()

    # Task: Get the i-th label. Since it's one number, .item() will work.
    label_index = # --- YOUR CODE HERE --- .item()

    plt.imshow(image)
    plt.title(f"Label: {label_index}\n({path_label_map[str(label_index)]})")
    plt.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

---
## Module 3: Data Preprocessing and Augmentation

Great job! Now that we understand our data, we need to prepare it for our AI model.

**1. Normalization (Preprocessing):** Think of this like a recipe. If one ingredient is measured in grams and another in pounds, it's confusing. Normalization converts all pixel values to a common scale (usually with a mean of 0 and a standard deviation of 1). This helps the model learn faster and more effectively.

**2. Data Augmentation:** What if you only have 10 pictures of a cat to learn from? You'd want to see that cat from different angles, in different lighting, etc. Data augmentation does this for our model. It creates new training data by randomly rotating, flipping, and shifting the original images. This prevents the model from just "memorizing" the training set (a problem called overfitting) and helps it generalize better to new, unseen images.

### Part A (Example): Preprocessing and Augmentation for `ChestMNIST`

A key part of normalization is knowing the dataset's mean and standard deviation. While some libraries provide these, it's a much better and more robust practice to calculate them ourselves directly from the training data. Let's do that now.

In [ ]:
# --- Calculate Mean and Std for ChestMNIST ---

# Load the training data with only ToTensor to get pixel values between 0 and 1
temp_dataset = medmnist.ChestMNIST(split='train', transform=transforms.ToTensor(), download=True)
# Use a DataLoader to iterate over the data in batches
temp_loader = DataLoader(dataset=temp_dataset, batch_size=1024, shuffle=False)

# Variables to store the sum of all pixel values and the sum of all squared pixel values
psum    = torch.tensor([0.0])
psum_sq = torch.tensor([0.0])

# Loop through the data
for inputs, _ in tqdm(temp_loader, desc="Calculating Mean/Std"):
    psum    += inputs.sum(axis=[0, 2, 3]) # sum over all axes except the channel axis
    psum_sq += (inputs**2).sum(axis=[0, 2, 3])

# Calculate the total number of pixels
count = len(temp_dataset) * 28 * 28

# Calculate the final mean and standard deviation
chest_mean = psum / count
chest_std  = torch.sqrt((psum_sq / count) - (chest_mean ** 2))

print(f"\nCalculated ChestMNIST Mean: {chest_mean.tolist()}")
print(f"Calculated ChestMNIST Std: {chest_std.tolist()}")

Now that we have the mean and standard deviation, let's create our final transformation pipelines and visualize the effect of augmentation.

In [ ]:
# This is our augmentation pipeline
chest_augmentation_transform = transforms.Compose([
    transforms.RandomRotation(5),      # Rotate by up to 5 degrees
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)), # Shift horizontally and vertically
    transforms.ToTensor(),             # Convert to Tensor
    transforms.Normalize(mean=chest_mean, std=chest_std) # Normalize with our calculated values
])

# The non-augmented pipeline for validation and testing
chest_data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=chest_mean, std=chest_std)
])


# Let's visualize the effect
original_image_pil, _ = medmnist.ChestMNIST(split='train', download=True)[0]

plt.figure(figsize=(12, 5))
plt.subplot(1, 5, 1)
plt.imshow(original_image_pil, cmap='gray')
plt.title('Original')
plt.axis('off')

# Show 4 augmented versions
for i in range(4):
    # Apply the augmentation pipeline directly to the PIL image
    augmented_tensor = chest_augmentation_transform(original_image_pil)

    # Un-normalize for visualization. This is a common pattern.
    un_normalize = transforms.Normalize(
        mean=[-m/s for m, s in zip(chest_mean, chest_std)],
        std=[1/s for s in chest_std]
    )
    unnormalized_tensor = un_normalize(augmented_tensor)

    plt.subplot(1, 5, i + 2)
    # Convert tensor back to a format matplotlib can show
    plt.imshow(unnormalized_tensor.permute(1, 2, 0).squeeze(), cmap='gray')
    plt.title(f'Augmented {i+1}')
    plt.axis('off')

plt.show()

### Part B (Project): Your Augmentation Pipeline for `PathMNIST`

Histopathology images are a perfect case for more aggressive data augmentation. Since the tissue can be placed on the slide in any orientation, random flips are very useful.

#### Your Turn: Calculate Mean/Std and Create an Augmentation Pipeline

First, you need to calculate the mean and standard deviation for the `PathMNIST` dataset, just like we did in the example. Remember that PathMNIST has 3 color channels, so you should get 3 values for the mean and 3 for the standard deviation.

After that, create a transformation pipeline for the **training set** that:
1.  Applies a random horizontal flip.
2.  Applies a random vertical flip.
3.  Converts the image to a Tensor.
4.  Normalizes the image using the mean and standard deviation you just calculated.

Then, create a separate, simpler pipeline for the **validation and test sets**. This one should *only* convert the image to a tensor and normalize it.

In [ ]:
# --- YOUR CODE HERE ---

# Task 1: Create a temporary dataset and dataloader for PathMNIST with only the ToTensor transform.
temp_path_dataset = # --- YOUR CODE HERE ---
temp_path_loader = # --- YOUR CODE HERE ---

# Task 2: Initialize tensors to store the sums. Remember PathMNIST has 3 channels!
psum    = # --- YOUR CODE HERE ---
psum_sq = # --- YOUR CODE HERE ---

# Task 3: Loop through the temp_path_loader, like in the ChestMNIST example, to calculate psum and psum_sq.
# --- YOUR CODE HERE ---

# Task 4: Calculate the final mean and standard deviation.
count = len(temp_path_dataset) * 28 * 28
path_mean = # --- YOUR CODE HERE ---
path_std  = # --- YOUR CODE HERE ---

print(f"\nCalculated PathMNIST Mean: {path_mean.tolist()}")
print(f"Calculated PathMNIST Std: {path_std.tolist()}")


# Task 5: Define the training transformation pipeline using your calculated mean and std.
path_train_transform = transforms.Compose([
    # --- YOUR CODE HERE ---
    # --- YOUR CODE HERE ---
    transforms.ToTensor(),
    transforms.Normalize(mean=path_mean, std=path_std)
])

# Task 6: Define the validation/test transformation pipeline.
path_val_test_transform = transforms.Compose([
    # --- YOUR CODE HERE ---
    # --- YOUR CODE HERE ---
])

print("\nPathMNIST transformation pipelines created successfully!")

#### Your Turn: Visualize PathMNIST Augmentations

Now, let's add a visualization to confirm your `path_train_transform` is working. Complete the code below to show an original pathology image and four augmented versions of it.

In [ ]:
# Task: Visualize the effect of your PathMNIST augmentations
original_path_image_pil, _ = medmnist.PathMNIST(split='train', download=True)[0]

plt.figure(figsize=(12, 5))
plt.subplot(1, 5, 1)
plt.imshow(original_path_image_pil)
plt.title('Original')
plt.axis('off')

# Show 4 augmented versions
for i in range(4):
    # Task: Apply your path_train_transform to the original PIL image
    augmented_tensor = # --- YOUR CODE HERE ---

    # Task: Create an un-normalize transform for PathMNIST using path_mean and path_std
    un_normalize = # --- YOUR CODE HERE ---
    unnormalized_tensor = un_normalize(augmented_tensor)

    plt.subplot(1, 5, i + 2)
    # Task: Display the unnormalized tensor. Remember to permute the dimensions!
    plt.imshow(# --- YOUR CODE HERE ---)
    plt.title(f'Augmented {i+1}')
    plt.axis('off')

plt.show()

## Day 4 complete!

Nice work — today you learned how to load, explore, visualize, and preprocess two real
medical imaging datasets (ChestMNIST and PathMNIST), including building your own data
augmentation pipeline for PathMNIST.

**Tomorrow (Day 5)** you'll use everything you set up today to actually build and train a
neural network, then explore transfer learning and model interpretability (saliency maps).

Tomorrow's notebook starts with a short recap/setup section that rebuilds the key pieces from
today (like the PathMNIST normalization values) in case you're starting a fresh Colab session
— you did this work yourself today, so this is just restoring where you left off, not new
material.